In [44]:
import gcsfs 
fs = gcsfs.GCSFileSystem()
import json
from helper import helpers as hp
import pandas as pd
from tqdm import tqdm

In [16]:
import importlib

# Data Loading

In [12]:
kpi_concept_meta_root = "gs://inno-twin-d-st-euwe1-data/US_FOOD/Concept_Kpi_meta/"
respondent_level_meta_root = "gs://inno-twin-d-st-euwe1-data/US_FOOD/0217_wave3/Respondent_meta/"
local_data_root = "./data/"

In [10]:
# get the concept file
with fs.open(kpi_concept_meta_root + "cid_concept.json") as f:
    cid_concept = json.load(f)
# get the respondent level data 
response_log = hp.load_jsonl(local_data_root + "0316_train_remove_extra_prompt.jsonl")

In [20]:
# df_response
response_table_df = pd.read_excel(respondent_level_meta_root + 'response_table_wave3.xlsx')
response_table_df = response_table_df.astype(str)



#  organized normal answer and interview
with fs.open(respondent_level_meta_root + '0206id_normal_answer_no_free_call.json', 'r', encoding='utf-8') as f:
    id_normal_answer = json.load(f)
with fs.open(respondent_level_meta_root + '0212_interview_no_probing_insight.json', 'r', encoding='utf-8') as f:
    id_interview = json.load(f)
id_normal_interview = {}
for k, v in id_normal_answer.items():
    try:
        normal_answer = id_normal_answer[k]
        interview_answer = id_interview[k] 
        merged_answer = normal_answer + "\n\n" + "--interview insights--" + "\n\n" + interview_answer
        id_normal_interview[k] = merged_answer
    except KeyError:
        continue
#  Clean based on the intersection of the respondent id
id_set = set(id_normal_answer.keys()) & set(id_interview.keys())
id_normal_answer = {k: v for k, v in id_normal_answer.items() if k in id_set}
id_interview = {k: v for k, v in id_interview.items() if k in id_set}
response_table_df = response_table_df.loc[response_table_df['ids'].isin(id_set)]
response_table_df.reset_index(drop=True, inplace=True)

response_table_df = response_table_df.astype(str)
response_table_df.drop('id', axis=1, inplace=True)
response_table_df.rename(columns={'ids':'id'}, inplace=True)

# rephrasing

In [39]:
import importlib
from augmentation import rephrasing
importlib.reload(rephrasing)

from augmentation.rephrasing import ConceptRephraser, RephraseConfig, Tone, PointOfView, ContentOrder

# Initialize the rephraser with the API key from the module
rephraser = ConceptRephraser(model="gpt-4o-mini", temperature=0.7)

In [ ]:
# Example: Rephrase a single concept with different configurations

# Pick a sample concept from cid_concept
sample_cid = list(cid_concept.keys())[1]
sample_text = cid_concept[sample_cid]
print(f"Original Concept (ID: {sample_cid}):")
print(sample_text)
print(f"\nWord count: {len(sample_text.split())}")
print("=" * 60)

# Example 1: Conversational tone + Second Person + Benefit-First
config_conversational = RephraseConfig(
    change_length=False,
    tone=Tone.CONVERSATIONAL,
    point_of_view=PointOfView.SECOND_PERSON,
    content_order=ContentOrder.BENEFIT_FIRST,
)

result_conv = rephraser.rephrase(sample_text, config_conversational)
print("\n[CONVERSATIONAL + Second Person + Benefit-First]:")
print(result_conv.rephrased_text)
print(f"\nWord count: {result_conv.original_word_count} → {result_conv.new_word_count}")
print("=" * 60)

# Example 2: Clinical tone + Third Person + Problem-First
config_clinical = RephraseConfig(
    change_length=False,
    tone=Tone.CLINICAL,
    point_of_view=PointOfView.THIRD_PERSON,
    content_order=ContentOrder.PROBLEM_FIRST,
)

result_clinical = rephraser.rephrase(sample_text, config_clinical)
print("\n[CLINICAL + Third Person + Problem-First]:")
print(result_clinical.rephrased_text)
print(f"\nWord count: {result_clinical.original_word_count} → {result_clinical.new_word_count}")
print("=" * 60)

# Example 3: Marketing tone + Length change + Feature-First
config_marketing = RephraseConfig(
    change_length=True,  # Will toggle between short/standard
    tone=Tone.MARKETING,
    point_of_view=PointOfView.SECOND_PERSON,
    content_order=ContentOrder.FEATURE_FIRST,
)

result_mkt = rephraser.rephrase(sample_text, config_marketing)
print("\n[MARKETING + Length Changed + Feature-First]:")
print(result_mkt.rephrased_text)
print(f"\nLength: {result_mkt.original_length_category.value} → {result_mkt.target_length_category.value}")
print(f"Word count: {result_mkt.original_word_count} → {result_mkt.new_word_count}")

In [42]:
import random

def multi_rephrase_concept(concept_text, n_variations=5):
    """
    Generate n random variations of a concept using different rephrasing configurations.
    
    Args:
        concept_text: The original concept text
        n_variations: Number of variations to generate (default: 5)
    
    Returns:
        List of dicts with 'config' and 'text' keys
    """
    tones = [None, Tone.CLINICAL, Tone.MARKETING, Tone.CONVERSATIONAL]
    povs = [None, PointOfView.SECOND_PERSON, PointOfView.THIRD_PERSON]
    orders = [None, ContentOrder.PROBLEM_FIRST, ContentOrder.FEATURE_FIRST, ContentOrder.BENEFIT_FIRST]
    length_options = [True, False]
    
    # Generate unique random configurations
    seen_configs = set()
    variations = []
    
    while len(variations) < n_variations:
        config_tuple = (
            random.choice(tones),
            random.choice(povs),
            random.choice(orders),
            random.choice(length_options)
        )
        
        # Skip if we've already used this exact config
        if config_tuple in seen_configs:
            continue
        seen_configs.add(config_tuple)
        
        tone, pov, order, change_len = config_tuple
        
        config = RephraseConfig(
            change_length=change_len,
            tone=tone,
            point_of_view=pov,
            content_order=order,
        )
        
        result = rephraser.rephrase(concept_text, config)
        
        variations.append({
            'config': {
                'tone': tone.value if tone else None,
                'point_of_view': pov.value if pov else None,
                'content_order': order.value if order else None,
                'change_length': change_len,
            },
            'text': result.rephrased_text,
            'word_count': result.new_word_count,
        })
    
    return variations


# Test it
sample_cid = list(cid_concept.keys())[0]
sample_text = cid_concept[sample_cid]
print(f"Original ({len(sample_text.split())} words):\n{sample_text}\n")
print("=" * 60)

variations = multi_rephrase_concept(sample_text, n_variations=5)

for i, var in enumerate(variations, 1):
    print(f"\n[Variation {i}] Config: {var['config']}")
    print(f"Word count: {var['word_count']}")
    print(var['text'])
    print("-" * 60)

Original (89 words):
OIKOS TRIPLE ZERO PLAIN HIGH PROTEIN YOGURT

More of what you want, less of what you don't - 18g of protein 0% fat, 0g added sugars and 0 artificial sweeteners.

Available in 32oz large size, Oikos Triple Zero Plain is a deliciously simple way to get the protein you need. Perfect to add to smoothies, parfaits, or enjoy on it's own!

- 18g Protein
- 0g Added Sugar
- 0 Artificial Sweeteners
- 0% Fat
- Project Non-GMO Verified

32oz Multi-Serve Tub - $5.99

Current Oikos Assortment Still Available


[Variation 1] Config: {'tone': 'marketing', 'point_of_view': 'third_person', 'content_order': 'feature_first', 'change_length': True}
Word count: 150
Introducing the revolutionary high-protein yogurt that is changing the game: Oikos Triple Zero Plain High Protein Yogurt! Packed with a remarkable 18g of protein, this yogurt is a powerhouse of nutrition that delivers more of what users crave while eliminating what they don’t. With 0% fat, 0g of added sugars, and absolutely n

In [45]:
from tqdm import tqdm

variation_cid_concept = {}

for k, v in tqdm(cid_concept.items(), desc="Processing concepts"):
    variations = multi_rephrase_concept(v, n_variations=5)
    sub_variations = {i+1: var for i, var in enumerate(variations)}
    variation_cid_concept[k] = sub_variations

Processing concepts: 100%|██████████| 12/12 [03:08<00:00, 15.75s/it]


In [52]:
with fs.open(kpi_concept_meta_root + "augmentation/variation_cid_concept.json", 'w') as f:
    json.dump(variation_cid_concept, f, indent=2)